In [ ]:
from firecrawl import AsyncFirecrawlApp

import os
from dotenv import load_dotenv

In [2]:
dotenv_path = os.path.abspath(os.path.join(os.getcwd(), '..', '.env'))
load_dotenv(dotenv_path)

API_KEY = os.getenv("GOOGLE_API_KEY")
FIRECRAWL_KEY = os.getenv("FIRECRAWL_API_KEY")

In [3]:
url = 'https://cookiechips.com'

In [ ]:
async def main():
    app = AsyncFirecrawlApp(api_key=FIRECRAWL_KEY)
    response = await app.map_url(
        url=url,
        include_subdomains=True
    )
    return response

# asyncio.run(main())

In [16]:
app = AsyncFirecrawlApp(api_key=FIRECRAWL_KEY)
response = await app.map_url(
    url=url,
    include_subdomains=True
)
response.links

['https://cookiechips.com',
 'https://cookiechips.com/pages/our-story',
 'https://cookiechips.com/pages/baking',
 'https://cookiechips.com/pages/happy-bakers',
 'https://cookiechips.com/products/chocolate-chip',
 'https://cookiechips.com/products/cinnamon-sugar',
 'https://cookiechips.com/products/sprinkle-sugar-cookies',
 'https://cookiechips.com/products/peanut-butter',
 'https://cookiechips.com/products/bakers-bundle',
 'https://cookiechips.com/blogs/ftpk',
 'https://cookiechips.com/blogs/ftpk/large-batch-cookie-recipes',
 'https://cookiechips.com/blogs/ftpk/best-baking-blog',
 'https://cookiechips.com/blogs/ftpk/chewy-chocolate-chip',
 'https://cookiechips.com/blogs/ftpk/charcuterie-board',
 'https://cookiechips.com/blogs/ftpk/gift-ideas',
 'https://cookiechips.com/blogs/ftpk/womans-world',
 'https://cookiechips.com/blogs/ftpk/crispy-edges-soft-middles',
 'https://cookiechips.com/blogs/ftpk/baking-and-cooking-with-kids',
 'https://cookiechips.com/blogs/ftpk/chocolate-truffles',
 'h

In [5]:
links = await main()
links = links.links
links

['https://cookiechips.com',
 'https://cookiechips.com/products/chocolate-chip',
 'https://cookiechips.com/products/cinnamon-sugar',
 'https://cookiechips.com/products/sprinkle-sugar-cookies',
 'https://cookiechips.com/products/peanut-butter',
 'https://cookiechips.com/products/bakers-bundle',
 'https://cookiechips.com/pages/our-story',
 'https://cookiechips.com/pages/baking',
 'https://cookiechips.com/pages/happy-bakers',
 'https://cookiechips.com/blogs/ftpk',
 'https://cookiechips.com/blogs/ftpk/large-batch-cookie-recipes',
 'https://cookiechips.com/blogs/ftpk/best-baking-blog',
 'https://cookiechips.com/blogs/ftpk/chewy-chocolate-chip',
 'https://cookiechips.com/blogs/ftpk/charcuterie-board',
 'https://cookiechips.com/blogs/ftpk/gift-ideas',
 'https://cookiechips.com/blogs/ftpk/womans-world',
 'https://cookiechips.com/blogs/ftpk/crispy-edges-soft-middles',
 'https://cookiechips.com/blogs/ftpk/baking-and-cooking-with-kids',
 'https://cookiechips.com/blogs/ftpk/chocolate-truffles',
 'h

In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import load_tools, initialize_agent, AgentType
from langchain.memory import ConversationBufferMemory

import re
import json


SERP_API_KEY = os.getenv('SERP_API_KEY')


llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", google_api_key=API_KEY, temperature=0)
memory = ConversationBufferMemory(memory_key='chat_history')
tools = load_tools(['serpapi','llm-math'], llm=llm, serpapi_api_key=SERP_API_KEY)
agent = initialize_agent(tools=tools,llm=llm, agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION, memory=memory,verbose=True, handle_parsing_errors=True)



filter_message = f"""
Eres un especialsita en recopilacion de datos de productos de tiendas online. Tu objetivo es a partir de una lista de urls definin cuales de los links son de productos de la tienda

Tu objetivo sera descartar todas las url que no sean de productos. Miraras en la lista de las urls y escojeras SOLO LAS QUE SEAN DE PRODUCTOS. Ten en cuenta que los productos pueden estar en varios idiomas

Es posible que exsistan productos repetidos en diferentes links o idiomas, deberas dercartar los productos que esten repetidos y quedarte solo con uno de ellos, si estan en diferentes idiomas escoje solo los que esten en español

Lista de URLs : 
{links}

La respuesta se usara para meter los datos en una base de datos asi que debe ser ÚNICAMENTE el siguiente objeto JSON, sin ningún texto antes o después, ni explicaciones, ni comentarios.

{{
"links" : [....]
}}

"""


data = agent.invoke(filter_message)['output']

json_match = re.search(r'\{.*\}', data, re.DOTALL)
if json_match:
    data = json_match.group(0)

data = json.loads(data)
product_links = data['links']

C:\Users\danie\AppData\Local\Temp\ipykernel_8656\4014285053.py:13: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(memory_key='chat_history')
C:\Users\danie\AppData\Local\Temp\ipykernel_8656\4014285053.py:15: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent = initialize_agent(tools=tools



> Entering new AgentExecutor chain...
Could not parse LLM output: ````json
{
"links": [
"https://cookiechips.com/products/chocolate-chip",
"https://cookiechips.com/products/cinnamon-sugar",
"https://cookiechips.com/products/sprinkle-sugar-cookies",
"https://cookiechips.com/products/peanut-butter",
"https://cookiechips.com/products/bakers-bundle"
]
}
````
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 
Observation: Invalid or incomplete response
Thought:Do I need to use a tool? No
AI: ```json
{
"links": [
"https://cookiechips.com/products/chocolate-chip",
"https://cookiechips.com/products/cinnamon-sugar",
"https://cookiechips.com/products/sprinkle-sugar-cookies",
"https://cookiechips.com/products/peanut-butter",
"https://cookiechips.com/products/bakers-bundle"
]
}
```

> Finished chain.


In [7]:
len(product_links)

5

In [8]:
app = AsyncFirecrawlApp(api_key=FIRECRAWL_KEY)
data = await app.batch_scrape_urls(
    product_links, 
    formats=['json'],
    json_options={
        'prompt' : 'Extract the title, sourceURL and Main Image from the page',
        'schema': {
            'type': 'json',
            'properties': {
                'title': {'type': 'string'},
                'sourceURL': {'type': 'string'},
                'imageURL' : {'type': 'string'}
            },
            'required': ['title', 'sourceURL', 'imageURL']
        }
    }
    )                                                                                                                                                                                              

In [9]:
scraped_webs = data.data
scraped_webs

[FirecrawlDocument(url=None, markdown=None, html=None, rawHtml=None, links=None, extract=None, json=None, screenshot=None, metadata={'favicon': 'https://cookiechips.com/cdn/shop/files/CookieChipsLinkenInCrowded_e8861408-0783-4d03-9efe-e62b0db36503.png?crop=center&height=32&v=1668458652&width=32', 'og:image:width': '778', 'title': 'American Express', 'google-site-verification': ['6VbQL9P3Y43JkIqKbASMcdN7rzD5ca8Kq_gbsdlZf5A', 'TZpyET5krjfQaUfo9yI_InHBGY-9YA8f0W_RdGziQLI'], 'og:image:height': '1024', 'shopify-checkout-api-token': '846c0a8b3bba1fe02aba15dce5020086', 'og:image': 'https://cdn.shopify.com/s/files/1/1670/6745/files/chocolate-chip-cookie-mix.jpg?v=1682631130', 'og:image:secure_url': 'https://cdn.shopify.com/s/files/1/1670/6745/files/chocolate-chip-cookie-mix.jpg?v=1682631130', 'og:image:alt': 'simple chocolate chip cookie mix', 'color-scheme': 'light dark', 'viewport': ['width=device-width,initial-scale=1', 'width=device-width, initial-scale=1.0,\n                              

In [10]:
products_message = f"""
Eres un especialsita en recopilacion de datos de productos de tiendas online. Tu objetivo es a partir de una lista de paginas de productos de un sitio web en markdown sacar toda la informacion necesaria de cada uno de los productos de la lista. Estan como FireCralw Document

Tu primer objetivo es descartar de las lista todas las paginas obsoletas o que vacias

Tu segundo objetivo es sacar la informacion de cada uno de los productos para esto tendras que leer los datos de todas las paginas de productos que has flitrado previamente

Que informacion sacaras?:
1. Nombre del Producto
2. Url del producto (es el sourceURL de la pagina)
3. Link a la imagen del producto (estara en algun "og:image:, habra varios links, asegurate que sea el link correcto)
4. Industria en la que se desarrolla el producto (Cosmetica, Salud, Telefonia, Suplementos, etc...)

Lista de paginas : 
{scraped_webs}

La respuesta se usara para meter los datos en una base de datos asi que debe ser ÚNICAMENTE el siguiente objeto JSON, sin ningún texto antes o después, ni explicaciones, ni comentarios:

{{
"producto-1": {{
        "nombre-producto": "...",
        "url-producto": "...",
        "url-imagen" : "...",
        "industria" : "..."
        }}
"producto-2": {{
        "nombre-producto": "...",
        "url-producto": "...",
        "url-imagen" : "...",
        "industria" : "..."
        }}
...
}}


El JSON devuelvelo sin saltos de linea, lo necescito asi

"""


data = agent.invoke(products_message)['output']

json_match = re.search(r'\{.*\}', data, re.DOTALL)
if json_match:
    data = json_match.group(0)

data = json.loads(data)




> Entering new AgentExecutor chain...
Could not parse LLM output: ````json
{
"producto-1": {
        "nombre-producto": "Chocolate Chip Cookie Mix",
        "url-producto": "https://cookiechips.com/products/chocolate-chip",
        "url-imagen" : "https://cdn.shopify.com/s/files/1/1670/6745/files/chocolate-chip-cookie-mix.jpg?v=1682631130",
        "industria" : "Alimentos"
        },
"producto-2": {
        "nombre-producto": "Cinnamon Sugar Cookie Mix",
        "url-producto": "https://cookiechips.com/products/cinnamon-sugar",
        "url-imagen" : "https://cdn.shopify.com/s/files/1/1670/6745/files/sugar-cookies-recipe-mix.jpg?v=1682614315",
        "industria" : "Alimentos"
        },
"producto-3": {
        "nombre-producto": "Peanut Butter Cookie Mix",
        "url-producto": "https://cookiechips.com/products/peanut-butter",
        "url-imagen" : "https://cdn.shopify.com/s/files/1/1670/6745/files/peanut-butter-cookies.jpg?v=1682614347",
        "industria" : "Alimentos"
      

In [11]:
data

{'producto-1': {'nombre-producto': 'Chocolate Chip Cookie Mix',
  'url-producto': 'https://cookiechips.com/products/chocolate-chip',
  'url-imagen': 'https://cdn.shopify.com/s/files/1/1670/6745/files/chocolate-chip-cookie-mix.jpg?v=1682631130',
  'industria': 'Alimentos'},
 'producto-2': {'nombre-producto': 'Cinnamon Sugar Cookie Mix',
  'url-producto': 'https://cookiechips.com/products/cinnamon-sugar',
  'url-imagen': 'https://cdn.shopify.com/s/files/1/1670/6745/files/sugar-cookies-recipe-mix.jpg?v=1682614315',
  'industria': 'Alimentos'},
 'producto-3': {'nombre-producto': 'Peanut Butter Cookie Mix',
  'url-producto': 'https://cookiechips.com/products/peanut-butter',
  'url-imagen': 'https://cdn.shopify.com/s/files/1/1670/6745/files/peanut-butter-cookies.jpg?v=1682614347',
  'industria': 'Alimentos'},
 'producto-4': {'nombre-producto': 'Sprinkle Sugar Cookie Mix',
  'url-producto': 'https://cookiechips.com/products/sprinkle-sugar-cookies',
  'url-imagen': 'https://cdn.shopify.com/s/f